# Suite PAT — Patterns: symbols, variables, constraints, locators, expressions and collectors

A pattern (`Schemas.OfPattern`) is the shape of a frame (docs/STREAMING_QUERIES.md): its symbols, each standing for a
reference object and perhaps located by a locator, a value of the `Locators` schemas (docs/OBJECT_IDENTITY.md,
Locators), its variables, each standing for a value, and the constraints a frame meets, expressions of boolean result.
An expression (`Schemas.OfExpression`) ranges over a pattern; its terms are mbse-expressions'. A collector
(`Schemas.OfCollector`) folds a pattern's frames. Each is a schema object, built, validated and written as every one is;
frames, and matching, are the executor's (step 4e of the store overhaul).

In [ ]:
from mbse.Schemas.Framework import Builders as B, JSON, Plain, Schemas as S
from support import raises


def wts(store):
    """wts.Features, a singleton directory owning definitions through `dir`, keyed by name, each owning its clauses
    through `clauses`; and wts.Tags, a relation of three links, from a definition: built in `store`."""
    Text = store.Schemas.OfStr.as_schema()
    Directory = store.Schemas.OfObject(lambda o: o.name("wts.Directory").ref(True).singleton("wts.Features"))
    Definition = store.Schemas.OfObject(lambda o: o.name("wts.Definition").ref(True).properties([
        lambda t: t.name("title").schema([Text]),
    ]))
    Clause = store.Schemas.OfObject(lambda o: o.name("wts.Clause").ref(True))
    Defines = store.Schemas.OfRelation(lambda r: (
        r
        .name("wts.Defines")
        .links([lambda t: t.name("directory"), lambda t: t.name("defn")])
        .properties([lambda t: t.name("name").schema([Text])])
        .uniques([["directory", "name"], ["defn"]])
    ))
    HasClause = store.Schemas.OfRelation(lambda r: (
        r
        .name("wts.HasClause")
        .links([lambda t: t.name("defn"), lambda t: t.name("clause")])
        .properties([lambda t: t.name("name").schema([Text])])
        .uniques([["defn", "name"], ["clause"]])
    ))
    Tags = store.Schemas.OfRelation(lambda r: r.name("wts.Tags").links([lambda t, n=n: t.name(n) for n in ("defn", "tag", "by")]))
    store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.name("dir").relation([Defines]).me("directory")]))
    store.Schemas.OfObject(Definition, lambda o: o.adjacencies([
        lambda t: t.name("def").relation([Defines]).me("defn").owner(True),
        lambda t: t.name("clauses").relation([HasClause]).me("defn"),
        lambda t: t.name("tags").relation([Tags]).me("defn"),
    ]))
    store.Schemas.OfObject(Clause, lambda o: o.adjacencies([lambda t: t.relation([HasClause]).me("clause").owner(True)]))
    return Directory, Definition, Clause, Defines, HasClause, Tags


store = B.Store()
Text, Int, Bool = (store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema())
Directory, Definition, Clause, Defines, HasClause, Tags = wts(store)

## PAT-01 · A pattern: symbols, variables and constraints, in order, each its own; validated

In [ ]:
Edit = store.Schemas.OfPattern(lambda p: (
    p
    .name("wts.Edit")
    .description("A definition and one of its clauses")
    .symbols([
        lambda t: t.name("defn").schema([Definition]),
        lambda t: t.name("clause").schema([Clause]).description("C-1"),
    ])
    .variables([lambda t: t.name("title").schema([Text]).description("its new title")])
    .requires([lambda c: c.description("the title is new")])
))
assert Edit.validate() == [] and list(Edit.symbols) == ["defn", "clause"] and list(Edit.variables) == ["title"]
assert Edit.variables["title"].schema is Text and Edit.symbols["clause"].description == "C-1"
assert len(Edit.requires) == 1 and Edit.requires[0].result is Bool  # a constraint: an expression of the store's OfBool
assert store.registered("wts.Edit") is Edit and store.module("wts").schemas["Edit"] is Edit  # named: in its module
twin = B.OfPattern.Builder(Edit, True, store).name("wts.Twin").clone()
assert twin.symbols["defn"] is not Edit.symbols["defn"] and twin.requires[0] is not Edit.requires[0]  # a clone owns its own
assert twin.variables["title"] is not Edit.variables["title"] and twin.variables["title"].schema is Text
Port = store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("pin").schema([Text])]))
bad = store.Schemas.OfPattern(lambda p: (
    p
    .symbols([lambda t: t.name("x").schema([Port]), lambda t: t.name("v")])
    .variables([lambda t: t.name("v").schema([Definition]), lambda t: t.name("w")])
    .parameters([lambda t: t.name("w")])
))
bad.variables["w"].description = 1  # what the builders refuse
assert bad.validate() == [
    "names used as both symbol and variable: ['v']",
    "names used as both variable and parameter: ['w']",
    "symbol 'x': a symbol stands for a reference object, so its schema is a reference object schema",
    "variable 'v': a reference object schema cannot be a property's schema",
    "variable 'w': a description is text, got int"]
assert S.OfPattern.Data(symbols={"p": S.OfSymbol.Data("p")}, parameters={"p": S.OfParameter.Data("p")}).validate() == [
    "names used as both symbol and parameter: ['p']"]

## PAT-02 · Locators: the shorthand, a path language, writes Locators data from a singleton or a symbol, hop by hop

In [ ]:
Located = store.Schemas.OfPattern(lambda p: p.name("wts.Located").symbols([
    lambda t: t.name("features").schema([Directory]).locator(lambda l: l.singleton(Directory)),
    lambda t: t.name("defn").schema([Definition]).locator(lambda l: l.symbol("features", Directory).dir({"name": "DEF-693"})),
    lambda t: t.name("clause").schema([Clause]).locator(
        lambda l: l.singleton(store.wts.Directory).dir({"name": "DEF-693"}).clauses({"name": "C-1"})),
    lambda t: t.name("any").schema([Clause]).locator(lambda l: l.symbol("defn", Definition).clauses()),  # no keys: each
]))
assert Located.validate() == []
root = Located.symbols["features"].locator
assert isinstance(root, S.Locators.OfSingleton.Data) and root.schema is Directory and root.next is None
from_symbol = Located.symbols["defn"].locator
assert isinstance(from_symbol, S.Locators.OfSymbol.Data) and (from_symbol.name, from_symbol.schema) == ("features", Directory)
hop = from_symbol.next
assert isinstance(hop, S.Locators.OfRelation.Data) and hop.next is None
assert (hop.schema, hop.adjacency, hop.link, hop.properties) == (Defines, "dir", "defn", {"name": "DEF-693"})
second = Located.symbols["clause"].locator.next.next  # the object after a hop has one schema: its next hop is found
assert (second.schema, second.adjacency, second.link, second.properties) == (HasClause, "clauses", "clause", {"name": "C-1"})
assert Located.symbols["any"].locator.next.properties == {}
# In full: a relation of more than two links, or after an object with no single schema.
tagged = store.Schemas.OfPattern(lambda p: p.symbols([
    lambda t: t.name("defn").schema([Definition]),
    lambda t: t.name("tag").locator(lambda l: l.symbol("defn", Definition).hop("tags", Tags, "tag", {})),
]))
assert tagged.validate() == [] and tagged.symbols["tag"].locator.next.link == "tag"
for path, says in [
    (lambda l: l.symbol("defn", Definition).tags(), "'wts.Tags' has more than two links: give the hop in full"),
    (lambda l: l.symbol("defn", Definition).hop("tags", Tags, "tag").dir(), "the object before 'dir' has no single schema"),
    (lambda l: l.singleton(Directory).nope(), "'wts.Directory' has no adjacency 'nope': a locator traverses named adjacencies"),
    (lambda l: l.hop("dir", Defines, "defn"), "a locator starts at a symbol or a singleton"),
    (lambda l: l.nothing, "the object before 'nothing' has no single schema"),
]:
    with raises(TypeError, AttributeError, ValueError, match=says):
        store.Schemas.OfSymbol(lambda s: s.locator(path))
with raises(ValueError, match="a locator starts at a symbol or a singleton"):
    B._LocatorPath(store)._built()
with raises(AttributeError, match="_private"):
    B._LocatorPath(store)._private
assert repr(B._LocatorPath(store)) == "<locator>"
assert B._LocatorPath(store).singleton(Directory).hop("dir", Defines, "nope")._current is None  # no such link: nothing after
assert B._LocatorPath(store).singleton(Directory).hop("dir", Directory, "x")._current is None  # nor through no relation
# The member is the symbol's `locator` adjacency too, as every adjacency is: linked, visited, or written in full.
built = store.Schemas.OfSymbol(lambda s: s.locator([root]))
assert built.locator is root
seen = []
store.Schemas.OfSymbol(built, lambda s: s.locator(lambda a: seen.append([e.locator.as_schema().name for e in a]) or a.as_iterable(lambda i: i)))
assert seen == [["Locators.OfSingleton"]]
store.Schemas.OfSymbol(built, lambda s: s.locator(lambda a: a.remove(lambda e: e)))
assert built.locator is None
direct = store.Locators.OfRelation(lambda h: h.schema([Defines]).adjacency("dir").link("defn").properties({"name": "D"}))
assert store.Locators.OfSymbol(lambda r: r.name("x").schema([Directory]).next([direct])).next is direct
assert B.Locators.OfRelation.Builder(direct, True, store).properties(None).clone().properties == {}  # cleared
assert B.Locators.OfSingleton.Builder is B.LocatorSingletonBuilder and S.Locators.OfSingleton.Builder is B.LocatorSingletonBuilder

## PAT-03 · What validation refuses of a locator: its root, its symbol, its singleton, each hop's relation, adjacency, link and properties

In [ ]:
def problems(locator, before=("defn",)):
    """A pattern's problems with a symbol `s` located by `locator`, after the symbols `before`, each a Definition."""
    symbols = {name: S.OfSymbol.Data(name, Definition) for name in before}
    symbols["s"] = S.OfSymbol.Data("s", locator=locator)
    return S.OfPattern.Data(symbols=symbols).validate()


Hop = S.Locators.OfRelation.Data
assert problems(Hop(Defines, "dir", "defn")) == [
    "symbol 's': a locator starts at a symbol or a singleton: Locators.OfSymbol or Locators.OfSingleton"]
assert problems(S.Locators.OfSymbol.Data("later", Definition)) == [
    "symbol 's': a locator starts at symbol 'later', which the pattern does not declare before it"]
assert problems(S.Locators.OfSymbol.Data("defn", Port)) == ["symbol 's': a locator's symbol has a reference object schema"]
assert problems(S.Locators.OfSingleton.Data(Definition)) == ["symbol 's': a locator's singleton is a singleton's schema"]
at = lambda hop: S.Locators.OfSingleton.Data(Directory, hop)
assert problems(at(S.Locators.OfSingleton.Data(Directory))) == [
    "symbol 's': hop 1: only a locator's root is a symbol or a singleton; each hop is a Locators.OfRelation"]
assert problems(at(Hop(Directory, "dir", "defn"))) == ["symbol 's': hop 1: a hop's schema is a relation"]
assert problems(at(Hop(HasClause, "dir", "defn"))) == [
    "symbol 's': hop 1: 'wts.Directory' has no adjacency 'dir' to 'wts.HasClause': a locator traverses named adjacencies"]
assert problems(at(Hop(Defines, "nope", "defn"))) == [
    "symbol 's': hop 1: 'wts.Directory' has no adjacency 'nope' to 'wts.Defines': a locator traverses named adjacencies"]
assert problems(at(Hop(Defines, "dir", "directory"))) == [
    "symbol 's': hop 1: 'directory' is not a link of 'wts.Defines' to the next object"]
assert problems(at(Hop(Defines, "dir", "defn", {"name": "D", "colour": "red"}))) == [
    "symbol 's': hop 1: 'wts.Defines' has no properties ['colour']"]
# After a hop whose link has no schema of its own, the adjacency is not checked; the hop after it still is.
assert problems(at(Hop(Defines, "dir", "defn", {}, Hop(Tags, "anything", "nope")))) == [
    "symbol 's': hop 2: 'nope' is not a link of 'wts.Tags' to the next object"]
link = S.OfRelation.Data(S.OfEntry.Data({"a": S.OfSymbol.Data("a"), "b": S.OfSymbol.Data("b", locator=at(None))}))
assert link.validate() == ["link 'b': a relation's link has no locator; a pattern's symbol may"]

## PAT-04 · Expressions and constraints: a pattern and a result schema; a constraint's result is OfBool; a pattern requires its own constraints

In [ ]:
Count = store.Schemas.OfExpression(lambda e: e.name("wts.Count").pattern([Edit]).result([Int]))
assert Count.validate() == [] and Count.pattern is Edit and Count.result is Int
assert store.registered("wts.Count") is Count and Count == Count and Count != Edit
Holds = store.Schemas.OfConstraint(lambda c: c.name("wts.Holds").pattern([Edit]))
assert Holds.result is Bool and Holds.validate() == [] and isinstance(Holds, S.OfExpression.Data)
assert B.OfConstraint.Builder().create().result.name == "Schemas.OfBool"  # outside a store, the template's
assert S.OfExpression.Data(pattern=Count).validate() == ["an expression's pattern is a pattern", "an expression needs a result schema"]
assert S.OfExpression.Data(result=Port, name="x y").validate() == [
    "name 'x y' is not identifiers separated by dots, e.g. 'crm.Contact'"]
assert S.OfExpression.Data(result=Definition).validate() == ["result: a reference object schema cannot be a property's schema"]
own = S.OfExpression.Data(result=Bool)
own.pattern = mine = S.OfPattern.Data(requires=(own,))
assert mine.validate() == []  # over its own pattern
assert S.OfPattern.Data(requires=(Count,)).validate() == [
    "requires[0] is a constraint: an expression whose result is Schemas.OfBool"]
assert S.OfPattern.Data(requires=(Holds,)).validate() == ["requires[0] is over this pattern's frames, not another's"]
assert S.OfPattern.Data(requires=(S.OfExpression.Data(result=Bool, description=2),)).validate() == [
    "requires[0]: a description is text, got int"]

## PAT-05 · Collectors: a pattern, the names it collects, commutative or associative, two suffixes, and what it computes

In [ ]:
Sum = store.Schemas.OfCollector(lambda c: (
    c
    .name("wts.Sum")
    .pattern([Edit])
    .symbols(["clause"])
    .variables(["title"])
    .commutative(True)
    .associative(True)
    .suffixes({"current": "_a", "next": "_b"})
    .computes([Count])
))
assert Sum.validate() == [] and (Sum.symbols, Sum.variables, Sum.suffixes) == (("clause",), ("title",), {"current": "_a", "next": "_b"})
assert B.OfCollector.Builder(Sum, False, store).symbols._held() == ["clause"] and Sum.computes is Count and Sum == Sum
cleared = B.OfCollector.Builder(Sum, True, store).name("wts.Cleared").symbols(None).variables(None).suffixes(None).clone()
assert (cleared.symbols, cleared.variables, cleared.suffixes) == ((), (), None)
assert S.OfCollector.Data().validate() == [
    "a collector needs a pattern",
    "a collector is commutative or associative, so that its result does not depend on the order frames arrive in"]
assert S.OfCollector.Data(Edit, ("nope",), ("none",), associative=True).validate() == [
    "it collects symbols ['nope'], which its pattern does not declare",
    "it collects variables ['none'], which its pattern does not declare"]
assert S.OfCollector.Data(Edit, commutative=True, suffixes={"current": "x"}).validate() == [
    "a collector's suffixes are 'current' and 'next'"]
assert S.OfCollector.Data(Edit, commutative=True, suffixes={"current": "x", "next": "x"}).validate() == [
    "a collector's 'current' and 'next' suffixes differ"]
assert S.OfCollector.Data(Edit, commutative=True, computes=Edit).validate() == ["what a collector computes is an expression"]
assert S.OfCollector.Data(Edit, commutative=True, computes=S.OfExpression.Data()).validate() == [
    "computes: an expression needs a result schema"]
assert B.OfCollector.Builder(store.Schemas.OfCollector(lambda c: c), False, store).suffixes._held() is None

## PAT-06 · Patterns, expressions, collectors and locators in a store's snapshot: written once, read back the same

In [ ]:
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log
assert JSON.loads(JSON.ToJSON(back).Snapshot()) == log
assert back.registered("wts.Located").symbols["clause"].locator.next.next.properties == {"name": "C-1"}
read = back.registered("wts.Edit")
assert read is not Edit and list(read.symbols) == ["defn", "clause"] and read.requires[0].result is back.Schemas.OfBool.as_schema()
assert back.registered("wts.Sum").pattern is read and back.registered("wts.Sum").computes is back.registered("wts.Count")
hops = [e for e in log if "exists" in e and e["exists"]["definition"].get("key", "").startswith("Locators.OfRelation/")]
assert len(hops) == 4 and all(e["exists"]["definition"]["properties"]["adjacency"] for e in hops)  # each hop, written once